In [22]:
import pandas as pd
from sqlalchemy import create_engine
from scipy.stats import kruskal
import scikit_posthocs as sp

files = {
    "orders" : "data/orders_dataset.csv",
    "reviews" : "data/reviews_dataset.csv",
}


In [23]:
engine = create_engine('sqlite:///olist.db')

for table, path in files.items():
    pd.read_csv(path).to_sql(table, engine, if_exists='replace', index=False)

query = """
SELECT o.order_id,
       o.order_delivered_customer_date,
       o.order_estimated_delivery_date,
       r.review_score
FROM orders o
JOIN reviews r
ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
"""

df = pd.read_sql(query, engine)
print(df.dtypes)
df

order_id                           str
order_delivered_customer_date      str
order_estimated_delivery_date      str
review_score                     int64
dtype: object


,order_id,order_delivered_customer_date,order_estimated_delivery_date,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-10 21:25:13,2017-10-18 00:00:00,4
1,53cdb2fc8bc7dce0b6741e2150273451,2018-08-07 15:27:45,2018-08-13 00:00:00,4
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-17 18:06:29,2018-09-04 00:00:00,5
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-12-02 00:28:42,2017-12-15 00:00:00,5
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-16 18:17:02,2018-02-26 00:00:00,5
...,...,...,...,...
96356,9c5dedf39a927c1b2549525ed64a053c,2017-03-17 15:08:01,2017-03-28 00:00:00,5
96357,63943bddc261676b46f01ca7ac2f7bd8,2018-02-28 17:37:56,2018-03-02 00:00:00,4
96358,83c1379a015df1e13d02aae0204711ab,2017-09-21 11:24:17,2017-09-27 00:00:00,5
96359,11c177c8e97725db2631073c19f07b62,2018-01-25 23:32:54,2018-02-15 00:00:00,2


In [24]:
estimated = pd.to_datetime(df['order_estimated_delivery_date'])
delivered = pd.to_datetime(df['order_delivered_customer_date'])

days_late = (delivered - estimated).dt.total_seconds() / 86400
df['days_late'] = days_late

bins = [float('-inf'), 0, 3, 7, 10, float('inf')]
labels = ['on-time', '1-3 days late','3-7 days late', '7-10 days late', '10+ days late']
df['late_bucket'] = pd.cut(days_late, bins=bins, labels=labels)

result = df.groupby('late_bucket',observed=True)['review_score'].agg(['mean','count'])
print(result)
df


                    mean  count
late_bucket                    
on-time         4.293718  88653
1-3 days late   3.765372   2651
3-7 days late   2.316263   1777
7-10 days late  1.799220   1026
10+ days late   1.699911   2246


,order_id,order_delivered_customer_date,order_estimated_delivery_date,review_score,days_late,late_bucket
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-10 21:25:13,2017-10-18 00:00:00,4,-7.107488,on-time
1,53cdb2fc8bc7dce0b6741e2150273451,2018-08-07 15:27:45,2018-08-13 00:00:00,4,-5.355729,on-time
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-17 18:06:29,2018-09-04 00:00:00,5,-17.245498,on-time
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-12-02 00:28:42,2017-12-15 00:00:00,5,-12.980069,on-time
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-16 18:17:02,2018-02-26 00:00:00,5,-9.238171,on-time
...,...,...,...,...,...,...
96356,9c5dedf39a927c1b2549525ed64a053c,2017-03-17 15:08:01,2017-03-28 00:00:00,5,-10.369433,on-time
96357,63943bddc261676b46f01ca7ac2f7bd8,2018-02-28 17:37:56,2018-03-02 00:00:00,4,-1.265324,on-time
96358,83c1379a015df1e13d02aae0204711ab,2017-09-21 11:24:17,2017-09-27 00:00:00,5,-5.524803,on-time
96359,11c177c8e97725db2631073c19f07b62,2018-01-25 23:32:54,2018-02-15 00:00:00,2,-20.018819,on-time


In [31]:
groups = [g['review_score'].dropna().values
for _, g in df.groupby('late_bucket', observed=True)]
stat, p = kruskal(*groups)
print(f"H = {stat:.4f}, p = {p}")

H = 10169.6393, p = 0.0


In [26]:
n = len(df)
epsilon_squared = stat / (n - 1)

if epsilon_squared < 0.01:
    effect = 'negligible'
elif epsilon_squared < 0.06:
    effect = 'small'
elif epsilon_squared < 0.14:
    effect = 'medium'
else:
    effect = 'large'

print(f'Epsilon-squared = {epsilon_squared:.3f} ({effect} effect)')

Epsilon-squared = 0.106 (medium effect)


In [27]:
clean = df.dropna(subset=['late_bucket', 'review_score']).copy()
clean['late_bucket'] = clean['late_bucket'].astype(str)

dunn = sp.posthoc_dunn(clean, val_col='review_score', group_col='late_bucket', p_adjust='bonferroni')
print(dunn)

def format_p(p):
    if p >= 0.05:
        return "ns"
    if p < 0.001:
        return "< .001"
    return f"{p:.3f}"

dunn.apply(lambda col: col.map(format_p))

                1-3 days late  10+ days late  3-7 days late  7-10 days late  \
1-3 days late    1.000000e+00   0.000000e+00  6.489242e-145   4.300418e-187   
10+ days late    0.000000e+00   1.000000e+00   5.667780e-26    1.000000e+00   
3-7 days late   6.489242e-145   5.667780e-26   1.000000e+00    2.543392e-12   
7-10 days late  4.300418e-187   1.000000e+00   2.543392e-12    1.000000e+00   
on-time         1.142971e-103   0.000000e+00   0.000000e+00    0.000000e+00   

                      on-time  
1-3 days late   1.142971e-103  
10+ days late    0.000000e+00  
3-7 days late    0.000000e+00  
7-10 days late   0.000000e+00  
on-time          1.000000e+00  


,1-3 days late,10+ days late,3-7 days late,7-10 days late,on-time
1-3 days late,ns,< .001,< .001,< .001,< .001
10+ days late,< .001,ns,< .001,ns,< .001
3-7 days late,< .001,< .001,ns,< .001,< .001
7-10 days late,< .001,ns,< .001,ns,< .001
on-time,< .001,< .001,< .001,< .001,ns
